# Same Split Fusion Experiments

Ce notebook reprend le script `same_split_fusion_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Fusion sequence+crop sur split coherent, base des politiques finales testees en streaming.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Same-parent-split sequence/crop fusion validation.
- Commande de reproduction referencee : same-split fusion.
- Artefacts controles : Same-parent-split fusion validation exists. (`runs/exp_020_same_split_fusion/metrics/same_split_fusion_summary.csv`).
- Run par defaut : `runs/exp_020_same_split_fusion`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "same_split_fusion_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path
from types import SimpleNamespace

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

from crop_cnn_experiments import CropDataset, TARGETS, predict, train_crop_model, video_metrics
from ml_pipeline import ROOT, safe_auc, threshold_sweep, write_json
from sequence_experiments import evaluate_catalogue_model, make_run_dir, train_one_model


## Fonction `normalize_for_split`

Cette cellule definit `normalize_for_split`. Elle prepare une partie du script.

In [ ]:
def normalize_for_split(X_raw, meta):
    train_mask = meta["split"].to_numpy() == "train"
    flat = X_raw[train_mask].reshape(-1, X_raw.shape[-1])
    mean = flat.mean(axis=0)
    std = flat.std(axis=0)
    std = np.where(std > 1e-6, std, 1.0)
    return ((X_raw - mean) / std).astype(np.float32), mean, std


## Fonction `make_parent_split`

Cette cellule definit `make_parent_split`. Elle prepare une partie du script.

In [ ]:
def make_parent_split(seq_meta, crop_index, seed):
    seq_video = seq_meta.groupby("video_id", as_index=False).agg(is_danger_clip=("is_danger_clip", "max"))
    crop_video = crop_index.groupby("video_id", as_index=False).agg(attention_label=("attention_label", "max"), blouse_label=("blouse_label", "max"))
    video = seq_video.merge(crop_video, on="video_id", how="left")
    video["attention_label"] = video["attention_label"].fillna(0).astype(int)
    video["blouse_label"] = video["blouse_label"].fillna(0).astype(int)
    video["combo"] = (
        video["is_danger_clip"].astype(int).astype(str)
        + "_"
        + video["attention_label"].astype(str)
        + "_"
        + video["blouse_label"].astype(str)
    )
    labels = video["combo"].to_numpy()
    if video["combo"].value_counts().min() < 3:
        labels = video["is_danger_clip"].astype(int).to_numpy()
    train_ids, temp_ids, _, temp_labels = train_test_split(
        video["video_id"].to_numpy(),
        labels,
        test_size=0.30,
        random_state=seed,
        stratify=labels,
    )
    temp_labels = np.asarray(temp_labels)
    stratify_temp = temp_labels
    counts = pd.Series(temp_labels).value_counts()
    if counts.min() < 2:
        stratify_temp = None
    val_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.50,
        random_state=seed + 1,
        stratify=stratify_temp,
    )
    split = {video_id: "train" for video_id in train_ids}
    split.update({video_id: "val" for video_id in val_ids})
    split.update({video_id: "test" for video_id in test_ids})
    return split


## Fonction `model_predict_positive`

Cette cellule definit `model_predict_positive`. Elle prepare une partie du script.

In [ ]:
def model_predict_positive(model, X):
    proba = model.predict_proba(X)
    classes = getattr(model, "classes_", None)
    if classes is None and hasattr(model, "named_steps"):
        classes = model.named_steps[list(model.named_steps.keys())[-1]].classes_
    if classes is None:
        return proba[:, 1]
    idx = list(classes).index(1)
    return proba[:, idx]


## Fonction `evaluate_fusion`

Cette cellule definit `evaluate_fusion`. Elle prepare une partie du script.

In [ ]:
def evaluate_fusion(df, score_col, seed, sequence_model, attention_arch, blouse_arch, persistence_windows):
    rows = []
    sweeps = []
    for split_name in ["train", "val", "test"]:
        split_df = df[df["split"] == split_name].copy()
        y = split_df["danger_within_1.0s"].astype(int).to_numpy()
        p = split_df[score_col].to_numpy()
        sweep = threshold_sweep(split_df.rename(columns={score_col: "risk"}), "risk", 1.0, split_name, persistence_windows=persistence_windows)
        sweep["repeat_seed"] = seed
        sweep["sequence_model"] = sequence_model
        sweep["fusion_variant"] = score_col
        sweeps.append(sweep)
        best = sweep.sort_values(["danger_clip_hit_rate", "safe_false_alarms_per_min", "window_precision"], ascending=[False, True, False]).iloc[0]
        rows.append(
            {
                "repeat_seed": seed,
                "sequence_model": sequence_model,
                "attention_arch": attention_arch,
                "blouse_arch": blouse_arch,
                "fusion_variant": score_col,
                "split": split_name,
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                "best_threshold_by_hit_fa": float(best["threshold"]),
                "best_hit_rate": float(best["danger_clip_hit_rate"]),
                "best_false_alarms_per_min": float(best["safe_false_alarms_per_min"]),
                "best_window_precision": float(best["window_precision"]),
                "best_window_recall": float(best["window_recall"]),
            }
        )
    return rows, sweeps


## Fonction `train_crop_predictions`

Cette cellule definit `train_crop_predictions`. Elle prepare une partie du script.

In [ ]:
def train_crop_predictions(run_dir, crop_index, seed, args, device):
    crop_args = SimpleNamespace(
        image_size=args.image_size,
        batch_size=args.crop_batch_size,
        lr=args.crop_lr,
        weight_decay=args.crop_weight_decay,
        epochs=args.crop_epochs,
        patience=args.crop_patience,
        no_pretrained=args.no_pretrained,
    )
    metrics = []
    history = []
    predictions = {}
    selected = {}
    for target, archs in [("attention", args.attention_architectures), ("blouse", args.blouse_architectures)]:
        target_rows = []
        for arch in archs:
            print(f"training crop seed{seed} {target} {arch}")
            model, hist, train_time_s, model_size = train_crop_model(run_dir, crop_index, target, arch, crop_args, device)
            model_path = run_dir / "models" / f"{target}_{arch}.pt"
            renamed = run_dir / "models" / f"{target}_seed{seed}_{arch}.pt"
            if model_path.exists():
                model_path.replace(renamed)
            for row in hist:
                row["repeat_seed"] = seed
            history.extend(hist)
            loader = DataLoader(
                CropDataset(run_dir, crop_index, target, train=False, image_size=args.image_size),
                batch_size=args.crop_batch_size,
                shuffle=False,
                num_workers=0,
            )
            probs, _ = predict(model, loader, device)
            pred = crop_index[["video_id", "split", "frame", "time_s", f"{target}_label"]].copy()
            pred["target"] = target
            pred["architecture"] = arch
            pred["repeat_seed"] = seed
            pred["risk"] = probs
            pred.to_csv(run_dir / "features" / f"crop_predictions_{target}_seed{seed}_{arch}.csv", index=False)
            predictions[(target, arch)] = pred
            for metric in video_metrics(pred, target):
                metric.update({"target": target, "architecture": arch, "repeat_seed": seed, "train_time_s": train_time_s, "model_size_bytes": model_size})
                target_rows.append(metric)
                metrics.append(metric)
        target_metrics = pd.DataFrame(target_rows)
        val = target_metrics[target_metrics["split"] == "val"].copy()
        val["score"] = val["average_precision"].fillna(0) + 0.2 * val["f1"].fillna(0)
        selected[target] = str(val.sort_values("score", ascending=False).iloc[0]["architecture"])
    return metrics, history, predictions, selected


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    sequence_run = Path(args.sequence_run)
    crop_run = Path(args.crop_run)
    if not sequence_run.is_absolute():
        sequence_run = ROOT / sequence_run
    if not crop_run.is_absolute():
        crop_run = ROOT / crop_run
    run_dir = make_run_dir(args.run_name)
    data = np.load(sequence_run / "features" / "sequence_dataset.npz")
    X_norm = data["X"].astype(np.float32)
    y = data["y"].astype(np.float32)
    mean = data["mean"].astype(np.float32)
    std = data["std"].astype(np.float32)
    X_raw = X_norm * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)
    seq_meta_base = pd.read_csv(sequence_run / "features" / "sequence_index.csv")
    crop_index_base = pd.read_csv(crop_run / "features" / "crop_cnn_index.csv")
    crop_index_base["path"] = crop_index_base["path"].apply(lambda p: str(crop_run / p))
    write_json(
        run_dir / "config.json",
        {
            "sequence_run": str(sequence_run),
            "crop_run": str(crop_run),
            "seeds": args.seeds,
            "sequence_specs": args.sequence_specs,
            "attention_architectures": args.attention_architectures,
            "blouse_architectures": args.blouse_architectures,
            "split_policy": "one parent-video split per seed shared by sequence and crop models",
        },
    )
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    seq_specs = []
    for spec in args.sequence_specs:
        name, kind, augment, loss = spec.split(":")
        seq_specs.append((name, kind, augment == "aug", loss))

    all_seq_metrics = []
    all_crop_metrics = []
    all_fusion_metrics = []
    all_sweeps = []
    all_history = []
    split_rows = []
    for seed in args.seeds:
        split = make_parent_split(seq_meta_base, crop_index_base, seed)
        seq_meta = seq_meta_base.copy()
        seq_meta["split"] = seq_meta["video_id"].map(split)
        crop_index = crop_index_base.copy()
        crop_index["split"] = crop_index["video_id"].map(split).fillna("unused")
        crop_index = crop_index[crop_index["split"] != "unused"].copy()
        seq_meta.to_csv(run_dir / "features" / f"sequence_index_seed{seed}.csv", index=False)
        crop_index.to_csv(run_dir / "features" / f"crop_index_seed{seed}.csv", index=False)
        split_rows.append({"repeat_seed": seed, **seq_meta.groupby("split")["video_id"].nunique().to_dict()})
        X, split_mean, split_std = normalize_for_split(X_raw, seq_meta)
        np.savez_compressed(run_dir / "features" / f"sequence_normalizer_seed{seed}.npz", mean=split_mean, std=split_std)

        crop_metrics, crop_history, crop_predictions, selected_crop = train_crop_predictions(run_dir, crop_index, seed, args, device)
        all_crop_metrics.extend(crop_metrics)
        all_history.extend(crop_history)
        att_pred = crop_predictions[("attention", selected_crop["attention"])]
        blouse_pred = crop_predictions[("blouse", selected_crop["blouse"])]
        att_map = att_pred.groupby("video_id")["risk"].mean().to_dict()
        ppe_map = blouse_pred.groupby("video_id")["risk"].mean().to_dict()

        for base_name, kind, augment, loss in seq_specs:
            model_name = f"seed{seed}_{base_name}"
            print(f"training sequence {model_name} {loss}")
            model_args = SimpleNamespace(
                seed=seed,
                batch_size=args.sequence_batch_size,
                lr=args.sequence_lr,
                weight_decay=args.sequence_weight_decay,
                epochs=args.sequence_epochs,
                patience=args.sequence_patience,
                loss=loss,
                label_smoothing=args.label_smoothing,
                focal_gamma=args.focal_gamma,
            )
            model, history, train_time_s, model_size = train_one_model(model_name, kind, augment, X, y, seq_meta, run_dir, model_args, device)
            for row in history:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_name
                row["model_family"] = "sequence"
            all_history.extend(history)
            seq_rows, _ = evaluate_catalogue_model(model_name, model, X, y, seq_meta, run_dir, device, train_time_s, model_size, args.sequence_batch_size, loss)
            for row in seq_rows:
                row["repeat_seed"] = seed
                row["base_architecture"] = base_name
            all_seq_metrics.extend(seq_rows)
            pred = pd.read_csv(run_dir / "features" / f"predictions_{model_name}.csv")
            df = pred.copy()
            df["danger_risk_1.0s"] = df["prob_1.0s"]
            df["attention_risk"] = df["video_id"].map(att_map).fillna(0.0)
            df["ppe_risk"] = df["video_id"].map(ppe_map).fillna(0.0)
            df["sequence_only"] = df["danger_risk_1.0s"]
            df["sequence_attention_rule"] = 1.0 - (1.0 - df["danger_risk_1.0s"]) * (1.0 - 0.30 * df["attention_risk"])
            df["sequence_ppe_rule"] = 1.0 - (1.0 - df["danger_risk_1.0s"]) * (1.0 - 0.25 * df["ppe_risk"])
            df["sequence_full_rule_fused"] = 1.0 - (1.0 - df["danger_risk_1.0s"]) * (1.0 - 0.30 * df["attention_risk"]) * (1.0 - 0.25 * df["ppe_risk"])
            meta_cols = ["danger_risk_1.0s", "attention_risk", "ppe_risk"]
            train = df[df["split"] == "train"].copy()
            meta_model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000, C=0.8))
            meta_model.fit(train[meta_cols], train["danger_within_1.0s"].astype(int))
            df["sequence_learned_meta_fused"] = model_predict_positive(meta_model, df[meta_cols])
            joblib.dump(
                {
                    "model": meta_model,
                    "meta_cols": meta_cols,
                    "sequence_model": model_name,
                    "attention_arch": selected_crop["attention"],
                    "blouse_arch": selected_crop["blouse"],
                },
                run_dir / "models" / f"meta_seed{seed}_{base_name}.joblib",
            )
            df.to_csv(run_dir / "features" / f"same_split_fused_seed{seed}_{base_name}.csv", index=False)
            for variant in ["sequence_only", "sequence_attention_rule", "sequence_ppe_rule", "sequence_full_rule_fused", "sequence_learned_meta_fused"]:
                rows, sweeps = evaluate_fusion(
                    df,
                    variant,
                    seed,
                    base_name,
                    selected_crop["attention"],
                    selected_crop["blouse"],
                    args.persistence_windows,
                )
                all_fusion_metrics.extend(rows)
                all_sweeps.extend(sweeps)
        pd.DataFrame(all_seq_metrics).to_csv(run_dir / "metrics" / "same_split_sequence_metrics.csv", index=False)
        pd.DataFrame(all_crop_metrics).to_csv(run_dir / "metrics" / "same_split_crop_metrics.csv", index=False)
        pd.DataFrame(all_fusion_metrics).to_csv(run_dir / "metrics" / "same_split_fusion_metrics.csv", index=False)
        pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "same_split_training_history.csv", index=False)
        if all_sweeps:
            pd.concat(all_sweeps, ignore_index=True).to_csv(run_dir / "metrics" / "same_split_fusion_threshold_sweeps.csv", index=False)

    fusion = pd.DataFrame(all_fusion_metrics)
    fusion.to_csv(run_dir / "metrics" / "same_split_fusion_metrics.csv", index=False)
    pd.DataFrame(all_seq_metrics).to_csv(run_dir / "metrics" / "same_split_sequence_metrics.csv", index=False)
    pd.DataFrame(all_crop_metrics).to_csv(run_dir / "metrics" / "same_split_crop_metrics.csv", index=False)
    pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "same_split_training_history.csv", index=False)
    pd.DataFrame(split_rows).to_csv(run_dir / "metrics" / "same_split_counts.csv", index=False)
    if all_sweeps:
        pd.concat(all_sweeps, ignore_index=True).to_csv(run_dir / "metrics" / "same_split_fusion_threshold_sweeps.csv", index=False)

    summary_rows = []
    for (sequence_model, variant, split_name), group in fusion.groupby(["sequence_model", "fusion_variant", "split"]):
        summary_rows.append(
            {
                "sequence_model": sequence_model,
                "fusion_variant": variant,
                "split": split_name,
                "n_repeats": int(group["repeat_seed"].nunique()),
                "ap_mean": float(group["average_precision"].mean()),
                "ap_std": float(group["average_precision"].std(ddof=0)),
                "hit_rate_mean": float(group["best_hit_rate"].mean()),
                "hit_rate_std": float(group["best_hit_rate"].std(ddof=0)),
                "false_alarms_per_min_mean": float(group["best_false_alarms_per_min"].mean()),
                "false_alarms_per_min_std": float(group["best_false_alarms_per_min"].std(ddof=0)),
                "precision_mean": float(group["best_window_precision"].mean()),
            }
        )
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(run_dir / "metrics" / "same_split_fusion_summary.csv", index=False)
    lines = ["# Same-Split Sequence/Crop Fusion", ""]
    lines.append("Each seed uses one parent-video split shared by sequence danger and crop attention/PPE models.")
    lines.append("")
    lines.append("| sequence | variant | split | AP mean | AP std | hit mean | FA/min mean | precision mean |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|")
    for _, row in summary.sort_values(["split", "ap_mean"], ascending=[True, False]).iterrows():
        lines.append(
            f"| {row['sequence_model']} | {row['fusion_variant']} | {row['split']} | {row['ap_mean']:.3f} | {row['ap_std']:.3f} | {row['hit_rate_mean']:.3f} | {row['false_alarms_per_min_mean']:.3f} | {row['precision_mean']:.3f} |"
        )
    (run_dir / "same_split_fusion_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Same-parent-split sequence/crop fusion validation.")
    parser.add_argument("--sequence-run", default="runs/exp_008_sequence_len60_catalogue")
    parser.add_argument("--crop-run", default="runs/exp_013_crop_cnn_catalogue")
    parser.add_argument("--run-name", default="exp_020_same_split_fusion")
    parser.add_argument("--seeds", nargs="+", type=int, default=[111, 222, 333])
    parser.add_argument(
        "--sequence-specs",
        nargs="+",
        default=["tcn_aug_focal:tcn:aug:focal", "tcn_aug_bce:tcn:aug:bce", "tcn_noaug_focal:tcn:noaug:focal"],
    )
    parser.add_argument("--attention-architectures", nargs="+", default=["small_cnn", "resnet18"])
    parser.add_argument("--blouse-architectures", nargs="+", default=["resnet18", "mobilenet_v3_small"])
    parser.add_argument("--sequence-epochs", type=int, default=20)
    parser.add_argument("--sequence-patience", type=int, default=5)
    parser.add_argument("--sequence-batch-size", type=int, default=128)
    parser.add_argument("--sequence-lr", type=float, default=1e-3)
    parser.add_argument("--sequence-weight-decay", type=float, default=1e-4)
    parser.add_argument("--crop-epochs", type=int, default=8)
    parser.add_argument("--crop-patience", type=int, default=3)
    parser.add_argument("--crop-batch-size", type=int, default=32)
    parser.add_argument("--crop-lr", type=float, default=3e-4)
    parser.add_argument("--crop-weight-decay", type=float, default=1e-4)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--label-smoothing", type=float, default=0.05)
    parser.add_argument("--focal-gamma", type=float, default=2.0)
    parser.add_argument("--persistence-windows", type=int, default=2)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_020_same_split_fusion_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["same_split_fusion_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
